[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GSTT-CSC/summer-school-tutorials/blob/day2/day2_qms_regs/QMS_runthrough_day2.ipynb)

# CSC Summer School Day 2 — Quality Management Systems & Regulations

## Clinical context: HandOsteo

**HandOsteo** is an as-yet unreleased Software as a Medical Device (SaMD) that automatically screens patients for osteoporosis and osteopenia from routine AP (anteroposterior) hand X-rays. It is currently in development by the GSTT-CSC Team

Osteoporosis is a skeletal disease characterised by reduced bone mineral density (BMD), affecting approximately 3.5 million people in the UK and causing around 500,000 fragility fractures per year.<sup>[[1]](#ref1)</sup> Early detection is critical — patients with known osteoporosis can be treated with bisphosphonates or lifestyle interventions that substantially reduce fracture risk.<sup>[[2]](#ref2)</sup>

A validated, low-cost screening proxy for BMD is the **Metacarpal Cortical Percentage (MCP)**:<sup>[[3]](#ref3)</sup> the ratio of cortical bone thickness to total bone width across the shaft of the second metacarpal, measured on a plain AP hand X-ray.

```
        |<------- A (total width) ------->|
        |<- B/2 ->|            |<- B/2 ->|
        ┌─────────┬────────────┬──────────┐
        │ cortex  │  medullary │  cortex  │
        └─────────┴────────────┴──────────┘
        
        MCP = ((A − B) / A) × 100
```

A MCP below 60 % is associated with osteopenia; below 50 % with osteoporosis.<sup>[[4,5]](#ref4)</sup>

### The HandOsteo pipeline

We have created a very basic copy of what the HandOsteo pipeline might look like:

| Step | Class | What it does |
|------|-------|--------------|
| 1 | `DicomLoader` | Load and validate the modality of the DICOM (must be CR or DX) |
| 2 | `ViewClassifier` | Confirm the view is AP or PA; reject oblique views |
| 3 | `MCPMeasurer` | Segment the second metacarpal and compute A, B, and MCP |
| 4 | `ReportGenerator` | Create a structured report and route it to PACS |

---

## What you will do today

You will act as both the developer and the quality engineer for HandOsteo and produce a lightweight **Quality Management System (QMS)**, based on the [GSTT CSC QMS template](https://github.com/GSTT-CSC/QMS-Template):

| # | Stage | What you will produce |
|---|-------|-----------------------|
| 1 | **Project Initiation** | Device description, classification, and role assignments |
| 2 | **Requirements Gathering** | System Requirements Specification (SRS) from the care process |
| 3 | **Hazard Log** | Risk assessment linked back to requirements |
| 4 | **Design Specification** | System Design Specification (SDS) items that implement each SRS |
| 5 | **Verification** | Automated unit tests that confirm the code meets the SDS |
| 6 | **Validation** | Manual test cases that confirm the system meets the SRS |

All artefacts are stored as YAML and rendered into QMS documents via a `make` pipeline — mirroring how the GSTT CSC QMS Template works.

---

### References

<a id="ref1"></a>1. Royal Osteoporosis Society. *Media Toolkit — Facts and Statistics*. Available at: https://theros.org.uk/about-us/media-centre/media-toolkit/

<a id="ref2"></a>2. National Institute for Health and Care Excellence. *Bisphosphonates for treating osteoporosis*. Technology appraisal TA464. London: NICE; 2018. Available at: https://www.nice.org.uk/guidance/ta464

<a id="ref3"></a>3. Barnett E, Nordin BEC. The radiological diagnosis of osteoporosis: a new approach. *Clin Radiol*. 1960;11(3):166–174. doi:10.1016/S0009-9260(60)80012-8

<a id="ref4"></a>4. Kim JW, Kim JK, Lim KH, et al. The 2nd metacarpal cortical index as a simple screening tool for osteopenia. *J Bone Metab*. 2020;27(4):261–266. doi:10.11005/jbm.2020.27.4.261 — [PMC7746479](https://pmc.ncbi.nlm.nih.gov/articles/PMC7746479/)

<a id="ref5"></a>5. Schreiber JJ, Anderson PA, Hsu WK. Simple assessment of global bone density and osteoporosis screening using standard radiographs of the hand. *J Hand Surg Am*. 2017;42(4):e272–e279. doi:10.1016/j.jhsa.2017.01.012 — [PMID 28242242](https://pubmed.ncbi.nlm.nih.gov/28242242/)

In [ ]:
# Make sure we have all our dependencies installed in Colab
%pip install -q rdm pandas pytest pyyaml pydicom ipytest pillow dicom2nifti nibabel svgwrite ipython

In [ ]:
import os
import sys
import yaml

# Check if we have the data and diagram helper files, and if not, clone them from the repo
if not os.path.isdir("day2_data") or not os.path.exists("src/care_process_diagram.py"):
    !git clone --depth=1 -b day2 https://github.com/GSTT-CSC/summer-school-tutorials.git _repo
    !cp -rn _repo/day2_qms_regs/day2_data .
    !cp -rn _repo/day2_qms_regs/src .
    !rm -rf _repo

# Add the current directory to the Python path if it's not already there
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Data ready:", os.path.isdir("day2_data"))
print("Diagram helper ready:", os.path.exists("src/care_process_diagram.py"))

# Create a custom YAML dumper to control indentation
class MyDumper(yaml.Dumper):
    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, False)

### HandOsteo Care Process Diagram

The diagram below shows a proposed care pathway for HandOsteo — automated osteoporosis/osteopenia screening from AP hand X-rays.

In [ ]:
from IPython.display import SVG, display
from src.care_process_diagram import build_diagram

display(SVG(data=build_diagram()))

> ### ✏️ How to edit files in Colab
>
> Several steps below ask you to edit a `.yml` file. In Colab:
>
> 1. Click the **📁 Files** icon in the left sidebar.
> 2. Open the **`day2_data/data/`** folder.
> 3. **Double-click** a file (e.g. `device.yml`) to open it in the editor pane on the right.
> 4. Make your edits, then press **Ctrl/Cmd-S** to save.
>
> ⚠️ Colab storage is temporary — if your runtime disconnects you'll lose your edits and the data, and need to re-run the setup cells above.

> ### 📐 About the worked examples
>
> Each data file already contains **one completed example** — but notice it describes a deliberately *unrelated* device: **EarthMeasurer**, Eratosthenes' 2,000-year-old method for estimating the circumference of the Earth from the angle of a shadow.
>
> That's on purpose. An unrelated example shows you the *shape* of a good requirement, hazard, design item, or test — without handing you the HandOsteo answers. Your job is to add the **HandOsteo** entries yourself, using the care process diagram above.
>
> Genuinely stuck? The full worked HandOsteo records are one toggle away 👇

> ### 💡 Stuck? Load the example solutions
>
> If you'd like to see a worked example for HandOsteo, run the cell below.
> It will overwrite the files in `day2_data/data/` with completed versions — you can then read through them
> and use them as a reference for the rest of the tutorial.
>
> ⚠️ This will replace anything you have already written in the data files.

In [ ]:
# 💡 Load the worked HandOsteo solutions into day2_data/data/
# Set LOAD_SOLUTIONS = True and run this cell only if you want to replace your work with the example answers.

LOAD_SOLUTIONS = False  # <-- change to True to load solutions

if LOAD_SOLUTIONS:
    import shutil
    from pathlib import Path

    for src in Path("day2_data/data_solutions").iterdir():
        shutil.copy(src, f"day2_data/data/{src.name}")
    print("Solutions loaded — refresh the file browser (↺) to see them.")
else:
    print(
        "LOAD_SOLUTIONS is False — nothing changed. Set it to True above if you want the example answers."
    )

# 1. Project Initiation

Open `day2_data/data/device.yml`. It currently holds the placeholder **EarthMeasurer** device — replace its details with **HandOsteo**'s:
- A **product name** and **version number**
- The **device classification** (class I, IIa, IIb, or III under UK MDR 2002)

Then open `day2_data/data/people.yml` and assign roles (Developer, Quality Engineer, Clinical Lead).

In [ ]:
# display the contents of the device and people yaml files.
with open("day2_data/data/device.yml", "r") as stream:
    device_data = yaml.safe_load(stream)

with open("day2_data/data/people.yml", "r") as stream:
    people_data = yaml.safe_load(stream)

print("Device Data:")
print(yaml.dump(device_data, Dumper=MyDumper, default_flow_style=False))
print("People Data:")
print(yaml.dump(people_data, Dumper=MyDumper, default_flow_style=False))

# 2. Requirements Gathering

Open `day2_data/data/requirements.yml` and add a new system requirement for HandOsteo, based on the care process diagram above. An example requirement (for the unrelated EarthMeasurer device) is already in the file to show the structure.

Each requirement needs:
- A short **title** and **description**
- A **unique identifier** (e.g. `SRS_002`)
- A **requirement type**

### Rules for writing good requirements

- **Clear, concise, and unambiguous** — one interpretation only
- **Measurable** — must be verifiable by an automated test
- Focus on **what** the software must do, not how it does it
- Use the word **MUST** (not "shall", which is ambiguous)

In [ ]:
# display the contents of the requirements yaml file.
with open("day2_data/data/requirements.yml", "r") as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

# 3. Hazard Log

Open `day2_data/data/risk.yml` and add a new hazard for HandOsteo based on the care process diagram.
An example hazard (again for the EarthMeasurer device) has been provided to show the structure.

### Rules

- **Hazard** — the potential source of harm (not the harm itself).
- **Causes** — what could lead to the hazardous situation.
- **Effects** — what happens as a result of the hazard.
- **Harm** — the impact on the patient or user.
- **Severity / Likelihood** — use the labels defined at the top of the file.
- **Risk control** — a mitigation that reduces the likelihood or severity to an acceptable level.
- Link your risk control back to a requirement using a `linked_risk_control` field in `requirements.yml`.

In [ ]:
# Display the contents of the risk yaml file
with open("day2_data/data/risk.yml", "r") as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

# 4. Design Specification

A requirement says *what* the software must do; a design specification says *how* you intend to satisfy it — in enough detail to build and test against.

Open `day2_data/data/requirements.yml` and add a design specification item (`sys_des_spec`) for the requirement you wrote in step 2. Link it back with the `linked_reqs` field so the requirement → design → test chain stays traceable.

A good design spec item:
- Has **enough detail** to implement the feature correctly
- Does **not** prescribe specific function names or class structures (unless a particular framework is required, e.g. "use MONAI")
- Is written so that it can be **verified by an automated test**

In [ ]:
# Display the contents of the requirements yaml file.
with open("day2_data/data/requirements.yml", "r") as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

### Checkpoint: render the QMS documents

The YAML files in `day2_data/data/` are the source records. The `make` command turns them into the controlled markdown documents in `day2_data/release/`.

Run the build cell, then preview each generated document. Check that your requirement, hazard, and SDS item read coherently as a traceable set before you start writing tests.

In [ ]:
# Build the documentation using the command 'make'
! cd day2_data && make && cd ..

In [ ]:
# 📄 See your completed QMS rendered into official documents
from IPython.display import Markdown, display

doc = "day2_data/release/system-requirements.md"
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print("Run the make cell above first to generate", doc)

In [ ]:
# 📄 See your completed QMS rendered into official documents
doc = "day2_data/release/hazard-log.md"
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print("Run the make cell above first to generate", doc)

In [ ]:
# 📄 See your completed QMS rendered into official documents
doc = "day2_data/release/system-design-specification.md"
if os.path.exists(doc):
    display(Markdown(open(doc).read()))
else:
    print("Run the make cell above first to generate", doc)

# 5. Verification

Verification asks a narrower question than validation: have we built the software according to the design specification?

You have already rendered your QMS documents. Now connect those documents to the behaviour of the software: first see how the current stub HandOsteo pipeline actually behaves, then write unit tests where each test verifies one SDS item and fails if the implementation no longer meets that design intent.

HandOsteo takes an AP hand X-ray (DICOM), classifies the view, measures the second metacarpal, and returns a report. We have provided a set of **stub classes** in `src/handosteo.py` that replicate the interface of the real system without needing a trained model. The stub is intentionally imperfect: it currently continues to produce measurements for oblique views, even though the design intent is to reject them.

We have 5 generated hand X-rays in `day2_data/xray/dicom/` — 3 AP views and 2 obliques. The APs are from patients with a range of bone health, while the obliques are from healthy patients. Each AP X-ray carries a **reference diagnosis** in its DICOM metadata (from a DEXA scan taken at the same visit), so we can compare expected clinical context, documented intent, and actual software behaviour.

> **Deliberate bug to catch:** leave the oblique-acceptance behaviour in place for now. The point is to design verification tests that describe the expected rejection behaviour and fail when the stub accepts an oblique view.

Run the next two cells to see the input images and how the stub handles each one.


In [ ]:
# Show sample hand X-rays: AP views (accepted) and oblique views (rejected)
from pathlib import Path
import pydicom
import matplotlib.pyplot as plt

xray_dir = Path("day2_data/xray/dicom")
files = {
    "AP hand 1": xray_dir / "AP_1.dcm",
    "AP hand 2": xray_dir / "AP_2.dcm",
    "AP hand 3": xray_dir / "AP_3.dcm",
    "Oblique hand 1": xray_dir / "Ob_1.dcm",
    "Oblique hand 2": xray_dir / "Ob_2.dcm",
}

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
for ax, (title, path) in zip(axes, files.items()):
    ds = pydicom.dcmread(str(path))
    arr = ds.pixel_array
    accepted = "Ob" not in path.name
    ax.imshow(arr, cmap="gray", aspect="equal")
    ax.set_title(
        title,
        fontsize=10,
        color="#1a7f1a" if accepted else "#c0392b",
        fontweight="bold",
    )
    ax.axis("off")
    border_color = "#1a7f1a" if accepted else "#c0392b"
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
        spine.set_visible(True)

fig.suptitle(
    "HandOsteo input data — AP views (green ✓) should be processed; oblique views (red ✗) should be rejected",
    fontsize=18,
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator
from pathlib import Path
import pandas as pd

loader = DicomLoader()
clf = ViewClassifier()
meas = MCPMeasurer()
rep_gen = ReportGenerator()

rows = []
for dcm_path in sorted(Path("day2_data/xray/dicom").glob("*.dcm")):
    ds = loader.load(str(dcm_path))
    reference = getattr(ds, "PatientComments", "—").replace("Reference diagnosis: ", "")
    try:
        view = clf.classify(ds)
        result = meas.measure(ds)
        report = rep_gen.generate(result)
        mcp = result["MCP"]
        outcome = report.decode().splitlines()[-1].replace("Classification: ", "")
        rows.append(
            {
                "File": dcm_path.name,
                "Reference (DEXA)": reference,
                "View": view,
                "MCP (%)": f"{mcp:.1f}",
                "HandOsteo says": outcome,
                "Status": "✓ ACCEPTED",
            }
        )
    except ValueError:
        rows.append(
            {
                "File": dcm_path.name,
                "Reference (DEXA)": reference,
                "View": getattr(ds, "ViewPosition", "?"),
                "MCP (%)": "—",
                "HandOsteo says": "—",
                "Status": "✗ REJECTED",
            }
        )

df = pd.DataFrame(rows)
display(df.to_html(index=False), metadata={"text/html": True})
print(df.to_string(index=False))

The pipeline above measures both AP and oblique views — the table shows obliques being accepted and given an MCP value when the design intent is to reject them. The tests you write next should make that failure visible.

### The pipeline classes

| Class | What it does |
|---|---|
| `DicomLoader` | Loads a DICOM file and validates its modality (CR or DX) |
| `ViewClassifier` | Classifies the view as AP, PA, or raises for oblique |
| `MCPMeasurer` | Measures A, B and calculates MCP = ((A−B)/A)×100 |
| `ReportGenerator` | Generates a byte-string report for success or failure |

### Test data

Synthetic hand X-ray DICOMs are in `day2_data/xray/dicom/`:

| File | View |
|---|---|
| `AP_1.dcm`, `AP_2.dcm`, `AP_3.dcm` | AP hand X-rays (should be accepted) |
| `Ob_1.dcm`, `Ob_2.dcm` | Oblique views (should be rejected) |

### What to test

For each SDS item, write at least one test that would **fail** if the implementation were broken. Think about:

- **Happy path** — does valid input produce the correct output?
- **Edge cases** — what boundary values matter?
- **Error paths** — what invalid inputs must be rejected, and with what exception?

Use `pytest.raises(...)` to assert that the right exceptions are raised.

The cell below zooms into a single oblique (`Ob_1.dcm`) from the table above to print the full report the stub produces — exactly the behaviour your rejection test should make fail.


In [ ]:
from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator
import ipytest

ipytest.autoconfig()

# Deliberate bug probe: oblique views should be rejected before measurement
loader = DicomLoader()
clf = ViewClassifier()
meas = MCPMeasurer()
rep_gen = ReportGenerator()

ds = loader.load("day2_data/xray/dicom/Ob_1.dcm")
view = clf.classify(ds)
result = meas.measure(ds)
report = rep_gen.generate(result)

print(f"View : {view}")
print(f"A    : {result['A']} mm")
print(f"B    : {result['B']} mm")
print(f"MCP  : {result['MCP']:.1f}%")
print(f"\nReport preview:\n{report.decode()}")

> ### ✏️ Your task
>
> Write at least one test per SDS item in the class below.
> Use `pytest.raises(SomeException)` to test error paths.
> Run the cell to see which tests pass.
>
> **Hint:** you can create a minimal in-memory DICOM dataset for mocking like this:
> ```python
> import pydicom
> ds = pydicom.Dataset()
> ds.Modality = "MG"   # wrong modality — loader should reject this
> ```

> ### 💡 Stuck? Load the example unit tests
>
> If you'd like to see a worked example, set `LOAD_SOLUTIONS = True` in the cell below and run it.
> The solution tests from `src/test_handosteo.py` will be run directly so you can read through them.
>
> ⚠️ Try to write your own tests first — the goal is to practise thinking like a verification engineer.

In [ ]:
# 💡 Load the worked HandOsteo unit tests
# Set LOAD_SOLUTIONS = True and run this cell to see the example test suite.

LOAD_SOLUTIONS = False  # <-- change to True

if LOAD_SOLUTIONS:
    from IPython.display import display, Markdown
    import subprocess

    # Show the solution source so students can read through it
    with open("src/test_handosteo.py") as f:
        src = f.read()
    display(Markdown(f"```python\n{src}\n```"))

    # Run the solution tests and print the results
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "src/test_handosteo.py", "-v"],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print(
        "LOAD_SOLUTIONS is False — nothing shown. Set it to True above to see the example tests."
    )

# 6. Validation

Verification (step 5) asked *"did we build the software **right**?"*. Validation asks the complementary question: *"did we build the **right** software?"* — does it meet the user's real clinical need, not just the written specification?

Add a manual validation test to `day2_data/data/manual_tests.yml` that partially satisfies one of your requirements.

### Rules for manual test cases

- Must be executable by an end user (not necessarily the developer)
- Steps must be clearly written with a defined expected outcome
- Should map back to a system requirement (`SRS_XXX`)

> **Note:** unlike the requirements, hazard log, and design spec, `manual_tests.yml` isn't rendered into a document by `make` — this lightweight tutorial ships no validation template in `day2_data/docs/`. It stays a YAML **source record**: the version-controlled data a validation report would be generated from in the full QMS-Template.

In [ ]:
# Display the contents of the manual validation tests yaml file.
with open("day2_data/data/manual_tests.yml", "r") as stream:
    data_loaded = yaml.safe_load(stream)

print(yaml.dump(data_loaded, Dumper=MyDumper, default_flow_style=False))

---
## 🎉 You've built a lightweight QMS

You've now produced and traced a complete (if lightweight) quality record for HandOsteo:

| Stage | Artefact | Source file |
|---|---|---|
| 1 · Project Initiation | Device record & roles | `device.yml`, `people.yml` |
| 2 · Requirements | System Requirements (SRS) | `requirements.yml` |
| 3 · Hazard Log | Risk assessment | `risk.yml` |
| 4 · Design Spec | System Design (SDS) | `requirements.yml` |
| 5 · Verification | Automated unit tests | your `TestHandOsteo` class |
| 6 · Validation | Manual test cases | `manual_tests.yml` |

Re-run the **`make`** cell at any time to regenerate the controlled documents in `day2_data/release/` (system requirements, hazard log, and system design specification) from your YAML source records — exactly how the [GSTT-CSC QMS-Template](https://github.com/GSTT-CSC/QMS-Template) turns structured data into a release-ready QMS.

Want to compare your work against a fully worked HandOsteo example? Flip `LOAD_SOLUTIONS = True` in the toggle cells and re-run.